# Opdracht 3 · Evaluatie diepgaand
### Classificatie-training · Antwoordmodel-versie

Verder kijken dan accuracy. Je gaat:

1. Een **confusion matrix** maken als heatmap met seaborn
2. **Precision en recall per klasse** bekijken
3. Inzoomen op het **lastige paar** versicolor vs. virginica
4. De **ROC-curve** tekenen en de AUC aflezen
5. De **beslissingsdrempel** schuiven en zien hoe precision en recall bewegen


## 0. Voorbereiding

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, classification_report,
                             precision_score, recall_score, roc_curve, roc_auc_score)

sns.set_theme(style="whitegrid")
print("Bibliotheken geladen.")

In [ ]:
# Data laden en splitsen
data = load_iris(as_frame=True)
df = data.frame.copy()
df["soort"] = df["target"].map({0:"setosa",1:"versicolor",2:"virginica"})
y = df["soort"]; X = df[data.feature_names]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
# Bewust een lastiger setup: alleen kelkblad-features (sepal), zodat we fouten zien om te bespreken
sepal_features = ["sepal length (cm)", "sepal width (cm)"]
model = LogisticRegression(max_iter=2000).fit(X_train[sepal_features], y_train)
y_pred = model.predict(X_test[sepal_features])
print("Model getraind op alleen kelkblad-features.")

## 1. Confusion matrix als heatmap
Met seaborn krijg je veel mooiere en leesbaardere matrices dan met de standaard scikit-learn-display.

### TODO 1 — Maak de confusion matrix
Bereken de confusion matrix met `confusion_matrix(y_test, y_pred, labels=...)`.

> Tip: geef `labels=["setosa","versicolor","virginica"]` mee zodat de volgorde vaststaat.

In [ ]:
# ANTWOORD
labels = ["setosa", "versicolor", "virginica"]
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels, cbar=False,
            linewidths=0.5, annot_kws={"size":13,"weight":"bold"})
plt.xlabel("Voorspelde klasse"); plt.ylabel("Werkelijke klasse")
plt.title("Confusion matrix (3 klassen)")
plt.show()

> **Bekijk de matrix:** waar zitten de fouten? Welke twee klassen worden met elkaar verward?

## 2. Precision en recall per klasse
`classification_report` geeft per klasse precision, recall en F1.

In [ ]:
# Classification report (al ingevuld)
print(classification_report(y_test, y_pred, digits=3))

> **Interpreteer:** voor welke klasse is recall laag? Wat betekent dat in de praktijk?

## 3. Inzoomen op het lastige paar
We beperken ons tot versicolor vs. virginica. Daar zit de echte trade-off tussen precision en recall.

### TODO 2 — Maak een binaire dataset
Filter `df` op rijen waarbij `soort` in `["versicolor", "virginica"]`, en maak een binaire target
`y_bin` die 1 is voor virginica en 0 voor versicolor.

In [ ]:
# ANTWOORD
mask = df["soort"].isin(["versicolor", "virginica"])
X_bin = df.loc[mask, data.feature_names]
y_bin = (df.loc[mask, "soort"] == "virginica").astype(int)

X_btr, X_bte, y_btr, y_bte = train_test_split(X_bin, y_bin, test_size=0.3, random_state=42, stratify=y_bin)

model_bin = LogisticRegression(max_iter=2000).fit(X_btr, y_btr)
y_proba = model_bin.predict_proba(X_bte)[:, 1]
print("Binair model getraind. Test-set:", len(y_bte), "bloemen.")

## 4. ROC-curve en AUC
De ROC toont de trade-off tussen true-positive-rate en false-positive-rate over alle drempels.

### TODO 3 — Bereken de AUC
Gebruik `roc_auc_score(y_bte, y_proba)`.

In [ ]:
# ANTWOORD
auc = roc_auc_score(y_bte, y_proba)
print(f"AUC: {auc:.3f}")

In [ ]:
# ROC-curve (al ingevuld)
fpr, tpr, thr = roc_curve(y_bte, y_proba)
plt.figure(figsize=(5.5, 4.5))
plt.plot(fpr, tpr, color="#4473C5", linewidth=2.5, label=f"model (AUC = {auc:.3f})")
plt.plot([0,1], [0,1], color="#9199B9", linestyle="--", label="gokken (AUC = 0,5)")
plt.xlabel("False-positive-rate"); plt.ylabel("True-positive-rate")
plt.title("ROC-curve  ·  versicolor vs. virginica")
plt.legend(loc="lower right")
plt.show()

## 5. De beslissingsdrempel schuiven
Standaard wordt 0.5 als drempel gebruikt: kans > 0.5 → virginica. Maar dat is een keuze.
We berekenen precision en recall voor verschillende drempels.

### TODO 4 — Bereken precision en recall voor verschillende drempels
Voor elke drempel in `drempels`, zet `y_proba > drempel` om naar een 0/1-voorspelling en bereken
precision en recall. De rest is al voor je geschreven.

In [ ]:
# ANTWOORD
drempels = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
resultaten = []
for d in drempels:
    y_pred_d = (y_proba > d).astype(int)
    p = precision_score(y_bte, y_pred_d, zero_division=0)
    r = recall_score(y_bte, y_pred_d)
    resultaten.append({"drempel": d, "precision": p, "recall": r})

resultaten = pd.DataFrame(resultaten)
print(resultaten.round(3))

In [ ]:
# Plot van precision en recall over de drempel (al ingevuld)
plt.plot(resultaten["drempel"], resultaten["precision"], marker="o", color="#4473C5", label="Precision")
plt.plot(resultaten["drempel"], resultaten["recall"], marker="s", color="#D97733", label="Recall")
plt.xlabel("Drempel"); plt.ylabel("Score")
plt.title("Precision en recall over de drempel")
plt.legend(); plt.show()

## 6. Bespreking
Bespreek met je buur:

- Welke twee klassen worden met elkaar verward in de confusion matrix? Komt dat overeen met de pairplot uit opdracht 1?
- Wat gebeurt er met precision en recall als je de drempel verhoogt?
- Stel je gebruikt dit model voor **spamdetectie** — kies je een hoge of lage drempel? Waarom?
- En voor **churn** (klanten die dreigen weg te gaan)? Welke drempel kies je?

---
### Toelichting voor de trainer
- Het model op alleen sepal-features haalt **~70-75% accuracy** — bewust een wat slechtere setup
  om confusion te zien. Setosa wordt nog wel goed gevonden, versicolor/virginica overlappen sterk.
- **AUC binair** is ~0.98 — heel hoog, omdat we wel weer alle features gebruiken voor het binaire model.
- **Drempel-tabel:** lage drempels geven hoge recall en lage precision; bij hoge drempels andersom.
  Spam → hoge drempel (precision belangrijk, geen vals alarm). Churn → lage drempel (recall belangrijk).